# GerryChain workflow: Gerrymandria cut edges

Run a 10,000-step ReCom chain and display the distribution of cut edges.

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

from gerrychain import Graph, MarkovChain, Partition
from gerrychain.updaters import Tally
from gerrychain.proposals import ReCom

from gerrytools.ben import RecordedChain
from gerrytools.plotting import Histogram
from gerrytools.scoring import PlanEvaluator, CutEdges, cut_edges

GRAPH_PATH = Path("../JSON_dualgraphs/gerrymandria.json")
RUN_NAME = "EXAMPLE_gerrymandria_cut_edges_walkthrough"
OUTPUT_PATH = Path(f"../chain_outputs/{RUN_NAME}.bendl")
STATS_DIR = Path(f"../stats/{RUN_NAME}")
POPULATION_COLUMN = "TOTPOP"
STARTING_PLAN = "district"
TOTAL_STEPS = 10_000
RNG_SEED = 42
POPULATION_TOLERANCE = 0.01

## Load the dual graph

In [ ]:
graph = Graph.from_json(str(GRAPH_PATH))
print(f"{len(graph.nodes)} nodes and {len(graph.edges)} edges")
graph.node_data(next(iter(graph.nodes)))

## Set up the Markov Chain

In [ ]:
chain = RecordedChain(
    graph,
    total_steps=TOTAL_STEPS,
    rng=RNG_SEED,
    output_path=OUTPUT_PATH
)

# Currently RecordedChains can only deal with integer assignments
assignment = {node: int(graph.node_data(node)[STARTING_PLAN]) for node in graph.nodes}

chain.initial_partition = Partition(
    chain.graph,
    assignment=assignment,
    updaters={
        "population": Tally(POPULATION_COLUMN),
    }
)

ideal_population = (
    sum(chain.initial_partition["population"].values()) / len(chain.initial_partition)
)

chain.proposal_fn = ReCom.district_pairs_mst(
    pop_col=POPULATION_COLUMN,
    pop_target=ideal_population,
    epsilon=POPULATION_TOLERANCE,
)

## Run 10,000 steps

In [ ]:
for partition in tqdm(chain.allow_overwrite(), total=TOTAL_STEPS):
    pass

## Create a Plan evaluator and re-evaluate the stream

In [ ]:
evaluator = PlanEvaluator(graph)
evaluator.add_metrics(CutEdges())
result = evaluator.evaluate_stream(OUTPUT_PATH, STATS_DIR, update=True)

## Display the cut-edge histogram

In [ ]:
histogram = Histogram(title="Gerrymandria ReCom cut edges")
histogram.add_dataset(result.read("cut_edges", return_type="series"))
histogram.add_vertical_lines(cut_edges(chain.initial_partition), linecolor="cherryblossompink")
histogram.set_bin_widths(1)
histogram.center_bars()
histogram.show()